# Feature Engineering - Customer Info

This notebook starts feature engineering for the customer segmentation project.

This phase uses only `customer_info` to create an unscaled, interpretable customer-level feature table. `customer_basket` features will be added later in a separate phase.

## Imports

Use direct data loading in the notebook and import reusable customer-info feature logic from `src/features.py`.

In [ ]:
import sys

import pandas as pd

sys.path.append("../src")

from features import (
    add_customer_info_features,
    create_basket_features,
    get_lifetime_spend_columns,
)

## Load Customer Info

Load the raw customer-level data directly from `data/raw`.

In [ ]:
customer_info = pd.read_csv("../data/raw/customer_info.csv")
customer_info.head()

## Reference Year Check

The raw maximum `year_first_transaction` is reviewed before calculating age and tenure. The maximum value is not used blindly because the initial EDA found future-looking years.

In [ ]:
year_reference_check = pd.DataFrame(
    {
        "metric": [
            "raw max year_first_transaction",
            "latest plausible transaction year used",
            "rows after 2026",
        ],
        "value": [
            customer_info["year_first_transaction"].max(),
            customer_info.loc[
                customer_info["year_first_transaction"] <= 2026,
                "year_first_transaction",
            ].max(),
            (customer_info["year_first_transaction"] > 2026).sum(),
        ],
    }
)

year_reference_check

The feature function uses the latest plausible transaction year in the data, `2026`, as the reference year for `age` and `customer_tenure`. The raw maximum is `2029`, so it is not treated as reasonable at this stage.

## Create Customer Info Features

Apply the reusable feature function to build the customer-level feature table.

In [ ]:
customer_features_info = add_customer_info_features(customer_info)
customer_features_info.head()

## Feature Columns

Review the generated spend share columns and the final feature table columns.

In [ ]:
lifetime_spend_columns = get_lifetime_spend_columns(customer_info)
spend_share_columns = [
    f"share_{column.replace('lifetime_spend_', '')}"
    for column in lifetime_spend_columns
]

print("Lifetime spend columns used:")
print(lifetime_spend_columns)
print("\nSpend share columns created:")
print(spend_share_columns)

In [ ]:
customer_features_info.columns.to_frame(index=False, name="column")

## Basic Validation

Check that the feature table remains one row per customer and keeps all customers from `customer_info`.

In [ ]:
feature_validation_checks = pd.DataFrame(
    {
        "check": [
            "customer_features_info rows",
            "customer_features_info columns",
            "unique customer_id count",
            "duplicated customer_id count",
            "one row per customer_id",
        ],
        "value": [
            customer_features_info.shape[0],
            customer_features_info.shape[1],
            customer_features_info["customer_id"].nunique(dropna=False),
            customer_features_info["customer_id"].duplicated().sum(),
            customer_features_info["customer_id"].is_unique,
        ],
    }
)

feature_validation_checks

In [ ]:
customer_features_info.head()

## Missing Values in Engineered Features

Inspect missing values in the newly engineered columns. Missing values are not fully resolved yet because preprocessing will happen later.

In [ ]:
engineered_feature_columns = [
    "age",
    "customer_tenure",
    "has_loyalty_card",
    "total_children_home",
    "has_children",
    "total_lifetime_spend",
    *spend_share_columns,
    "degree_level",
]

engineered_missing_summary = pd.DataFrame(
    {
        "missing_count": customer_features_info[engineered_feature_columns].isna().sum(),
        "missing_percentage": customer_features_info[engineered_feature_columns].isna().mean() * 100,
    }
).sort_values("missing_percentage", ascending=False)

engineered_missing_summary

## Summary Statistics

Review the new numerical features before saving the intermediate table.

In [ ]:
new_numerical_features = [
    "age",
    "customer_tenure",
    "has_loyalty_card",
    "total_children_home",
    "has_children",
    "total_lifetime_spend",
    *spend_share_columns,
]

customer_features_info[new_numerical_features].describe()

In [ ]:
customer_features_info["degree_level"].value_counts(dropna=False)

In [ ]:
customer_features_info["has_loyalty_card"].value_counts(dropna=False)

## Save Intermediate Customer Feature Table

Save the unscaled, interpretable customer-level feature table based only on `customer_info`. This is not a model-ready scaled dataset.

In [ ]:
customer_features_info.to_csv("../data/processed/customer_features_info.csv", index=False)
print("Saved ../data/processed/customer_features_info.csv")

## Notes

- `customer_name` is not kept raw because names are identifiers, not useful modelling inputs; only the simple `degree_level` prefix signal is retained.
- `loyalty_card_number` is not kept raw because it behaves like an indicator; it is represented as `has_loyalty_card`.
- Spend shares were created to describe each customer's spending mix relative to total lifetime spend.
- This table is not model-ready yet because missing values, invalid values, encoding, scaling, and feature selection still need to be handled.
- Basket-derived features will be added in the next phase.
- Preprocessing will happen later, after the full customer-level feature table is assembled.

# Customer Basket Feature Engineering

Add simple customer-level features derived from `customer_basket`. These features are based only on the sampled basket records and are merged into the existing customer-info feature table.

## Load Customer Basket and Customer Info Features

Load `customer_basket` directly from the raw data folder and reload the existing customer-info feature table from `data/processed`.

In [ ]:
customer_basket = pd.read_csv("../data/raw/customer_basket.csv")
customer_basket.head()

In [ ]:
customer_features_info = pd.read_csv("../data/processed/customer_features_info.csv")
customer_features_info.head()

## Create Basket-Derived Features

Use reusable functions from `src/features.py` to parse `list_of_goods`, compute basket sizes, and aggregate basket behavior by `customer_id`.

In [ ]:
basket_features = create_basket_features(customer_basket)
basket_features.head()

In [ ]:
basket_features.shape

## Merge Basket Features

Left merge basket features into the customer-info feature table so every customer remains in the final dataset.

In [ ]:
customer_features = customer_features_info.merge(basket_features, on="customer_id", how="left")

basket_numeric_features = [
    "basket_count",
    "avg_basket_size",
    "max_basket_size",
    "min_basket_size",
    "total_items_bought_in_baskets",
    "distinct_products_in_baskets",
    "avg_distinct_products_per_basket",
]

customer_features[basket_numeric_features] = customer_features[basket_numeric_features].fillna(0)

basket_count_columns = [
    "basket_count",
    "max_basket_size",
    "min_basket_size",
    "total_items_bought_in_baskets",
    "distinct_products_in_baskets",
]
customer_features[basket_count_columns] = customer_features[basket_count_columns].astype(int)

customer_features["most_frequent_product"] = customer_features["most_frequent_product"].fillna("No Basket")
customer_features.head()

## Basket Feature Validation

Check that the merged customer-level table preserves all customers and has no duplicate customer IDs.

In [ ]:
basket_feature_validation_checks = pd.DataFrame(
    {
        "check": [
            "basket_features rows",
            "basket_features columns",
            "customer_features rows",
            "customer_features columns",
            "customer_features row count equals 33,038",
            "unique customer_id count",
            "one row per customer_id",
            "duplicated customer_id count",
            "customers with basket_count == 0",
        ],
        "value": [
            basket_features.shape[0],
            basket_features.shape[1],
            customer_features.shape[0],
            customer_features.shape[1],
            customer_features.shape[0] == 33038,
            customer_features["customer_id"].nunique(dropna=False),
            customer_features["customer_id"].is_unique,
            customer_features["customer_id"].duplicated().sum(),
            (customer_features["basket_count"] == 0).sum(),
        ],
    }
)

basket_feature_validation_checks

In [ ]:
customer_features[basket_numeric_features].describe()

In [ ]:
customer_features["most_frequent_product"].value_counts(dropna=False).head(20)

In [ ]:
customer_features.isna().sum().sort_values(ascending=False)

## Save Final Customer Feature Table

Save the unscaled customer-level feature table with customer_info and customer_basket-derived features.

In [ ]:
customer_features.to_csv("../data/processed/customer_features.csv", index=False)
print("Saved ../data/processed/customer_features.csv")

## Basket Feature Notes

- Basket features are based on the sampled baskets only.
- Customers without basket records are kept and receive `0` for numeric basket features.
- `most_frequent_product` is useful for profiling, but may not be used directly in clustering.
- The dataset is still not preprocessed, scaled, or model-ready.
- Preprocessing and feature selection will happen later.